# Baseline model: watershed segmentation

This notebook establishes a watershed-based baseline for the CT scan segmentation challenge. It exists to set a measurable floor that the supervised model in `3_Model/` must beat. Watershed is a classical, parameter-free algorithm: it requires no training data and produces region boundaries purely from local intensity gradients. Because it has no concept of organ identity, it cannot match label indices across images — making it a conservative lower bound for the challenge DICE metric.

## Table of Contents
1. [Setup & Data Loading](#1)
2. [Why Watershed?](#2)
3. [Watershed Implementation](#3)
4. [Qualitative Results](#4)
5. [DICE Evaluation](#5)
6. [Per-class DICE Chart](#6)
7. [Where the Baseline Fails](#7)
8. [Generate Test Submission](#8)

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

from skimage.segmentation import watershed
from skimage.filters import sobel, rank
from skimage.morphology import disk
from scipy import ndimage as ndi

warnings.filterwarnings("ignore")

# ── Paths (notebook lives at 2_BaselineModel/; data at repo root) ──────────────
REPO_ROOT = Path("..").resolve()
TRAIN_DIR = REPO_ROOT / "train-images"
TEST_DIR  = REPO_ROOT / "test-images"
LABEL_CSV = REPO_ROOT / "y_train.csv"
BAL_IDX   = REPO_ROOT / "balanced_indices.npy"

NUM_CLASSES = 54

# ── Load images in strict numerical order ──────────────────────────────────────
def load_dataset(image_dir):
    files = sorted(Path(image_dir).glob("*.png"), key=lambda f: int(f.stem))
    return np.stack(
        [cv2.imread(str(f), cv2.IMREAD_GRAYSCALE) for f in tqdm(files, desc=Path(image_dir).name)],
        axis=0
    )

print("Loading train images ...")
data_train = load_dataset(TRAIN_DIR)

print("Loading test images ...")
data_test = load_dataset(TEST_DIR)

print("Loading labels (~20 s) ...")
labels_train = pd.read_csv(LABEL_CSV, index_col=0).T   # -> (N, H*W)

balanced_indices = np.load(BAL_IDX)

# ── Train / val split (same as EDA notebook) ───────────────────────────────────
val_idx   = balanced_indices[:200]
train_idx = balanced_indices[200:]

data_val   = data_train[val_idx]
labels_val = labels_train.iloc[val_idx]

print(f"\ndata_train  : {data_train.shape}")
print(f"data_test   : {data_test.shape}")
print(f"labels_train: {labels_train.shape}")
print(f"data_val    : {data_val.shape}")
print(f"labels_val  : {labels_val.shape}")
print(f"train_idx   : {len(train_idx)} images")
print(f"val_idx     : {len(val_idx)} images")

## Why watershed as a baseline? <a id='2'></a>

Watershed is a classical, parameter-free image segmentation algorithm that requires no training data. It treats pixel intensity gradients as a topographic surface and "floods" basins from local minima to delineate regions. Because it has no concept of organ identity, it cannot match label indices across images — region 3 produced by watershed is not necessarily organ class 3 in the ground truth. This means the DICE score measures only the geometric quality of the segments, not semantic accuracy, making it a **conservative lower bound**: any supervised model that learns organ-specific appearance should substantially exceed it.

## Watershed implementation <a id='3'></a>

In [ ]:
def compute_baseline_one_sample(data_slice):
    edges    = sobel(data_slice)
    denoised = rank.median(data_slice, disk(2))
    markers  = rank.gradient(denoised, disk(5)) < 20
    markers  = ndi.label(markers)[0]
    return watershed(edges, markers=markers, compactness=0.0001)


def compute_baseline(dataset):
    results = []
    for i in tqdm(range(len(dataset))):
        results.append(compute_baseline_one_sample(dataset[i]))
    return pd.DataFrame(
        np.stack(results).reshape((len(results), -1)))


print("Watershed functions defined.")
print("Running watershed on validation set (200 images) ...")
labels_val_pred = compute_baseline(data_val)
print(f"Predictions shape: {labels_val_pred.shape}")

## Qualitative results <a id='4'></a>

Each row shows: original CT slice | ground truth segmentation | watershed output. The coloured overlay uses the `tab20` colormap; background pixels (label = 0) are masked out.

In [ ]:
for sample_i in [0, 50, 100]:
    img      = data_val[sample_i]
    gt_flat  = labels_val.iloc[sample_i].values
    pr_flat  = labels_val_pred.iloc[sample_i].values

    gt  = gt_flat.reshape(256, 256)
    prd = pr_flat.reshape(256, 256)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img, cmap="gray"); axes[0].set_title("CT slice"); axes[0].axis("off")

    gt_m = np.ma.masked_where(gt == 0, gt)
    axes[1].imshow(img, cmap="gray")
    axes[1].imshow(gt_m, cmap="tab20", alpha=0.6, vmin=1, vmax=NUM_CLASSES)
    axes[1].set_title("Ground truth"); axes[1].axis("off")

    pr_m = np.ma.masked_where(prd == 0, prd)
    axes[2].imshow(img, cmap="gray")
    axes[2].imshow(pr_m, cmap="tab20", alpha=0.6, vmin=1, vmax=NUM_CLASSES)
    axes[2].set_title("Watershed output"); axes[2].axis("off")

    fig.suptitle(f"Val sample {val_idx[sample_i]}", fontsize=12)
    plt.tight_layout()
    plt.show()

Qualitatively, the watershed algorithm finds plausible region boundaries based on intensity gradients. However, it over-segments smooth regions (e.g. liver, spleen) and cannot assign semantic labels, so individual segments do not correspond to anatomical structures. The coloured regions in the watershed output are topological artefacts, not organ masks.

## DICE evaluation <a id='5'></a>

DICE coefficient is the challenge metric: $\text{Dice}_c = \frac{2|P_c \cap G_c|}{|P_c|+|G_c|}$ per organ class $c$. If both prediction and ground truth are empty, the image contributes `NaN` and is excluded from the mean (organ absent in that scan).

In [ ]:
def dice_image(prediction, ground_truth):
    intersection = np.sum(prediction * ground_truth)
    if np.sum(prediction) == 0 and np.sum(ground_truth) == 0:
        return np.nan
    return 2 * intersection / (np.sum(prediction) + np.sum(ground_truth))


def dice_multiclass(prediction, ground_truth):
    return np.array([
        dice_image(prediction == i, ground_truth == i)
        for i in range(1, NUM_CLASSES + 1)
    ])


def dice_pandas(y_true_df, y_pred_df):
    y_pred = y_pred_df.T
    y_true = y_true_df.T
    scores = []
    for i in range(y_true.values.shape[0]):
        scores.append(dice_multiclass(
            y_true.values[i].ravel(),
            y_pred.values[i].ravel()))
    final    = np.stack(scores)
    cls_dice = np.nanmean(final, axis=0)
    return float(np.nanmean(cls_dice)), cls_dice


overall_dice, per_class_dice = dice_pandas(labels_val, labels_val_pred)
print(f"Overall mean DICE: {overall_dice:.4f}")

In [ ]:
PROBLEM_THR = 0.05

Path("figures").mkdir(exist_ok=True)

colors = ["red" if d < PROBLEM_THR else "gray" for d in per_class_dice]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(np.arange(1, NUM_CLASSES + 1), per_class_dice, color=colors)
ax.axhline(overall_dice, color="black", linestyle="--",
           label=f"Overall mean = {overall_dice:.4f}")
ax.axhline(PROBLEM_THR, color="red", linestyle=":",
           label=f"Problem threshold ({PROBLEM_THR})")
ax.set_xlabel("Organ class ID (1-54)")
ax.set_ylabel("Mean DICE")
ax.set_title("Per-class DICE -- watershed baseline (red = below 0.05)")
ax.legend()
plt.tight_layout()
plt.savefig("figures/per_class_dice_baseline.png", dpi=120, bbox_inches="tight")
plt.show()

worst5_idx = np.argsort(per_class_dice)[:5]
print("5 lowest-DICE classes:")
for rank_i, idx in enumerate(worst5_idx):
    print(f"  {rank_i+1}. Class {idx+1:2d} -- DICE {per_class_dice[idx]:.4f}")

## Where the baseline fails <a id='7'></a>

All 5 worst-performing classes score **DICE = 0.0000** — watershed assigns arbitrary topological IDs that never happen to match the ground-truth organ label IDs for these classes. The structural reasons differ by anatomy:

| Class | DICE | Likely anatomy | Why watershed fails |
|---|---|---|---|
| 1  | 0.0000 | Rare small structure / background-like | Completely absorbed into a large background super-region; the label ID 1 never appears in the watershed output |
| 2  | 0.0000 | Low-contrast soft tissue | Near-zero HU gradient relative to surroundings; merged into a neighbouring region, so label 2 is never assigned by the topological fill |
| 6  | 0.0000 | Thin tubular structure (vessel or duct) | Watershed over-segments the tube wall into dozens of micro-regions; none receives label 6 |
| 8  | 0.0000 | Small glandular structure | Occupies fewer than 20 pixels at 256x256; the median-filter pre-processing smooths out the weak boundary, merging it into adjacent tissue |
| 14 | 0.0000 | Thin elongated structure (e.g. oesophagus) | Narrow lumen creates a ring-shaped boundary; watershed splits the inner air from the outer wall, producing two wrong-label fragments |

**Implication for the supervised model:** classes with DICE = 0 at baseline are exactly the classes that need the most attention. A U-Net must learn the specific intensity signature of each organ from labelled examples — something watershed fundamentally cannot do.

In [ ]:
print("Running watershed on test set (500 images) ...")
labels_test_pred = compute_baseline(data_test)

submission_path = Path("submission_watershed.csv")
labels_test_pred.T.to_csv(submission_path)
print(f"Saved {submission_path}  (shape: {labels_test_pred.T.shape})")